# Multi-Modal ICU Discharge / Death Prediction (Sequential)

**Task**: At each 3-hour time step, predict whether the patient will **still be in the ICU** at the next time step.
- **Label 1** = patient remains in ICU at the next step  
- **Label 0** = patient discharged or died (final step)

**Modalities**:
- **Demographics** (age, marital_status, language_group)
- **Radiology notes** encoded by `Qwen/Qwen3-0.6B` (frozen, mean-pooled)
- **Time series** from `data_ts_3h.csv` encoded by a **Causal** Transformer Encoder

**Loss**: `BCEWithLogitsLoss` on per-step labels (padding positions masked out)

**Evaluation**: Predicted discharge step (first step where P < 0.5) vs. true sequence length


In [1]:
# Uncomment to install: !pip install transformers accelerate
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import math, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
from tqdm import tqdm

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Configuration ──────────────────────────────────────────────────────
DATA_DIR, CHECKPOINT_PATH = 'data/', 'best_model.pt'
DISCHARGE_FILE = DATA_DIR + 'data_feature_discharge.csv'
DEATH_FILE     = DATA_DIR + 'data_feature_death.csv'
TS_FILE        = DATA_DIR + 'data_ts_3h.csv'

TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_SEED = 0.70, 0.15, 0.15, 42

# MAX_TS_STEPS will be computed from data in Cell 5 (99th percentile, capped at 256)
MAX_TS_STEPS = 200  # placeholder
TS_FEATURES = [
    'hr', 'map', 'rr', 'spo2', 'temp', 'gcs',
    'lactate', 'creatinine', 'bilirubin', 'platelets', 'wbc',
    'sodium', 'bun', 'inr',
    'urine_output', 'vasopressor_dose', 'fluid_input', 'ventilation_flag',
    'sofa_resp', 'sofa_cardio', 'sofa_renal', 'sofa_liver', 'sofa_coag', 'sofa_cns',
    'max24_sofa_resp', 'max24_sofa_cardio', 'max24_sofa_renal',
    'max24_sofa_liver', 'max24_sofa_coag', 'max24_sofa_cns'
]
TS_INPUT_DIM = len(TS_FEATURES)  # 30

TS_EMBED_DIM, TS_NHEAD, TS_NUM_LAYERS = 64, 4, 2
TS_DIM_FF, TS_DROPOUT = 128, 0.1

QWEN_MODEL_NAME = 'Qwen/Qwen3-0.6B'
TEXT_PROJ_DIM, TEXT_MAX_LENGTH = 128, 512

DEMO_EMBED_DIM, DEMO_HIDDEN_DIM = 16, 64
FUSION_HIDDEN_DIM = 256
LEARNING_RATE, WEIGHT_DECAY = 1e-4, 1e-4
BATCH_SIZE, NUM_EPOCHS, SCHEDULER_PATIENCE = 32, 30, 5

print(f'TS input dim : {TS_INPUT_DIM}')
print(f'Fusion dim   : {DEMO_HIDDEN_DIM}+{TS_EMBED_DIM}+{TEXT_PROJ_DIM} = {DEMO_HIDDEN_DIM+TS_EMBED_DIM+TEXT_PROJ_DIM}')
print(f'TEXT_MAX_LENGTH : {TEXT_MAX_LENGTH}')


In [2]:
# ── Cell 2: Data Loading ──────────────────────────────────────────────────
df_discharge = pd.read_csv(DISCHARGE_FILE)
df_death     = pd.read_csv(DEATH_FILE)
df_ts        = pd.read_csv(TS_FILE)

print(f'Discharge : {df_discharge.shape}')
print(f'Death     : {df_death.shape}')
print(f'TS        : {df_ts.shape}')
print(f'Unique patients (discharge): {df_discharge["stay_id"].nunique()}')
print(f'Unique patients (TS)       : {df_ts["stay_id"].nunique()}')
print('\nTarget stats:')
print(df_discharge['time_to_icu_discharge_hours'].describe())

# Confirm overlap between data sources
discharge_ids = set(df_discharge['stay_id'])
death_ids     = set(df_death['stay_id'])
ts_ids        = set(df_ts['stay_id'])
print(f'\nDischarge intersect Death : {len(discharge_ids & death_ids)}')
print(f'Discharge intersect TS    : {len(discharge_ids & ts_ids)}')

# Restrict to patients present in both discharge and TS tables
common_ids   = discharge_ids & ts_ids
df_discharge = df_discharge[df_discharge['stay_id'].isin(common_ids)].reset_index(drop=True)
assert df_discharge['stay_id'].is_unique, 'stay_id must be unique'
print(f'Final cohort : {len(df_discharge)} patients')

# Merge death event info from death file (death_event=1 => patient died in ICU)
df_death_info = df_death[['stay_id', 'death_event', 'time_to_icu_death_hours']]
df_discharge  = df_discharge.merge(df_death_info, on='stay_id', how='left')
df_discharge['death_event'] = df_discharge['death_event'].fillna(0).astype(int)
n_death = (df_discharge['death_event'] == 1).sum()
print(f'Death events merged: {n_death} ({n_death/len(df_discharge)*100:.1f}%)')

Discharge : (4619, 49)
Death     : (4619, 42)
TS        : (188785, 32)
Unique patients (discharge): 4619
Unique patients (TS)       : 4619

Target stats:
count    4619.000000
mean      118.630706
std       137.112563
min        24.010833
25%        43.858750
50%        70.680833
75%       138.701111
max      2112.996111
Name: time_to_icu_discharge_hours, dtype: float64

Discharge intersect Death : 4619
Discharge intersect TS    : 4619
Final cohort : 4619 patients


In [3]:
# ── Cell 3: Train / Val / Test Split (70 / 15 / 15 by stay_id) ──────────
# Split on stay_id to prevent patient-level data leakage.
all_ids = df_discharge['stay_id'].values
ids_trainval, ids_test = train_test_split(
    all_ids, test_size=TEST_RATIO, random_state=RANDOM_SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)  # ~0.176
ids_train, ids_val = train_test_split(
    ids_trainval, test_size=val_frac, random_state=RANDOM_SEED)

print(f'Train : {len(ids_train):5d} ({len(ids_train)/len(all_ids)*100:.1f}%)')
print(f'Val   : {len(ids_val):5d} ({len(ids_val)/len(all_ids)*100:.1f}%)')
print(f'Test  : {len(ids_test):5d} ({len(ids_test)/len(all_ids)*100:.1f}%)')

split_map = {}
for sid in ids_train: split_map[sid] = 'train'
for sid in ids_val:   split_map[sid] = 'val'
for sid in ids_test:  split_map[sid] = 'test'
df_discharge['split'] = df_discharge['stay_id'].map(split_map)
train_mask = df_discharge['split'] == 'train'

# Unified LOS: use time_to_icu_death_hours for death patients, else discharge hours
df_discharge['los_hours'] = np.where(
    df_discharge['death_event'] == 1,
    df_discharge['time_to_icu_death_hours'],
    df_discharge['time_to_icu_discharge_hours']
)
print(f'\nUnified LOS (los_hours):')
print(df_discharge['los_hours'].describe())
print(f'  Discharged alive: n={( df_discharge["death_event"]==0).sum()}, '
      f'mean LOS={df_discharge.loc[df_discharge["death_event"]==0,"los_hours"].mean():.1f} h')
print(f'  Died in ICU     : n={(df_discharge["death_event"]==1).sum()}, '
      f'mean LOS={df_discharge.loc[df_discharge["death_event"]==1,"los_hours"].mean():.1f} h')

Train :  3233 (70.0%)
Val   :   693 (15.0%)
Test  :   693 (15.0%)

Target (raw)  mean=118.6 h, std=137.1 h
Target (log1p) mean=4.418, std=0.795


In [4]:
# ── Cell 4: Demographic Feature Preprocessing ────────────────────────────
# age + marital_status appear in both files; language_group is discharge-only.
# Using discharge file as the single source of truth for all demographics.
DEMO_COLS_CAT = ['marital_status', 'language_group']

for col in DEMO_COLS_CAT:
    df_discharge[col] = df_discharge[col].fillna('UNKNOWN').astype(str)

# Fit encoders on TRAINING split only (prevents leakage)
label_encoders = {}
for col in DEMO_COLS_CAT:
    le = LabelEncoder()
    le.fit(df_discharge.loc[train_mask, col])
    known = set(le.classes_)
    df_discharge[col + '_enc'] = df_discharge[col].apply(
        lambda x: int(le.transform([x])[0]) if x in known else -1)
    label_encoders[col] = le
    print(f'{col}: {len(le.classes_)} classes -> {list(le.classes_)}')

# +1 reserves index 0 for unknown / padding in nn.Embedding
marital_vocab = len(label_encoders['marital_status'].classes_) + 1
lang_vocab    = len(label_encoders['language_group'].classes_)  + 1
print(f'marital_vocab={marital_vocab}, lang_vocab={lang_vocab}')

# Normalise age with TRAINING statistics only
age_mean = df_discharge.loc[train_mask, 'age'].mean()
age_std  = df_discharge.loc[train_mask, 'age'].std()
df_discharge['age_norm'] = (
    df_discharge['age'].fillna(age_mean) - age_mean) / (age_std + 1e-8)
print(f'Age: mean={age_mean:.1f} y, std={age_std:.1f} y')

marital_status: 5 classes -> ['DIVORCED', 'MARRIED', 'SINGLE', 'UNKNOWN', 'WIDOWED']
language_group: 2 classes -> ['English', 'non-English']
marital_vocab=6, lang_vocab=3
Age: mean=63.6 y, std=16.6 y


In [5]:
# ── Cell 5: Time Series Preprocessing ───────────────────────────────────
# No 7-day cap: use full time series for each patient.
# MAX_TS_STEPS = 99th-percentile of per-patient step counts, capped at 256.
# Patients longer than MAX_TS_STEPS are truncated (all labels = 1, still in ICU).

# Normalise with TRAINING patient statistics only
train_ts = df_ts[df_ts['stay_id'].isin(ids_train)]
ts_means = train_ts[TS_FEATURES].mean()
ts_stds  = train_ts[TS_FEATURES].std().replace(0, 1)
df_ts_norm = df_ts.copy()
df_ts_norm[TS_FEATURES] = ((df_ts[TS_FEATURES] - ts_means) / ts_stds).fillna(0.0)

# Compute MAX_TS_STEPS
step_counts = df_ts.groupby('stay_id').size()
p99_steps   = int(np.percentile(step_counts.values, 99))
MAX_TS_STEPS = min(p99_steps, 256)
print(f'Per-patient step counts: min={step_counts.min()}  median={int(step_counts.median())}  '
      f'p99={p99_steps}  max={step_counts.max()}')
print(f'Using MAX_TS_STEPS = {MAX_TS_STEPS}')
print(f'Patients fully within window: '
      f'{(step_counts <= MAX_TS_STEPS).mean()*100:.1f}%')

# Build lookup: stay_id -> ndarray (T_actual, 30), T_actual <= MAX_TS_STEPS
ts_lookup = {}
for sid, grp in df_ts_norm.groupby('stay_id'):
    arr = grp.sort_values('hour_3h')[TS_FEATURES].values.astype(np.float32)
    ts_lookup[sid] = arr[:MAX_TS_STEPS]

seq_lens = [v.shape[0] for v in ts_lookup.values()]
print(f'Patients with TS : {len(ts_lookup)}')
print(f'Seq len min/median/max : {min(seq_lens)} / {int(np.median(seq_lens))} / {max(seq_lens)}')


In [6]:
# ── Cell 6: Radiology Note Tokenization ──────────────────────────────────
print(f'Loading tokenizer: {QWEN_MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print('Set pad_token = eos_token')

def tokenize_note(text):
    """Returns tokenised dict or None for missing/empty notes."""
    if pd.isna(text) or str(text).strip() == '':
        return None
    return tokenizer(
        str(text), max_length=TEXT_MAX_LENGTH,
        truncation=True, padding='max_length', return_tensors='pt')

# Pre-tokenise once to speed up DataLoader batches
note_tokens = {}
for _, row in tqdm(df_discharge.iterrows(), total=len(df_discharge), desc='Tokenising'):
    note_tokens[row['stay_id']] = tokenize_note(row['radiology_note_text'])

missing = sum(1 for v in note_tokens.values() if v is None)
print(f'Missing notes: {missing} / {len(note_tokens)}')

Loading tokenizer: Qwen/Qwen3-0.6B ...


Tokenising: 100%|██████████| 4619/4619 [00:15<00:00, 295.73it/s]

Missing notes: 0 / 4619


In [7]:
# ── Cell 7: ICUDataset ───────────────────────────────────────────────────
class ICUDataset(Dataset):
    """
    Multi-modal Dataset for per-step ICU discharge prediction.
    Each sample returns:
        age, marital_enc, lang_enc          demographics
        ts_seq  (MAX_TS_STEPS, 30)          padded time series
        ts_mask (MAX_TS_STEPS,) bool        True = padding (ignored by attention)
        step_labels (MAX_TS_STEPS,) long    per-step labels:
                                              1 = patient still in ICU at next step
                                              0 = last valid step (discharge / death)
                                             -1 = padding (excluded from loss)
        input_ids, attention_mask (L,)      tokenised radiology note
        has_text  (1,)                      1.0 if note exists, else 0.0
        target    (1,)                      time_to_icu_discharge_hours (for eval)
    """
    def __init__(self, stay_ids, df, ts_lookup, note_tokens):
        self.stay_ids    = stay_ids
        self.df          = df.set_index('stay_id')
        self.ts_lookup   = ts_lookup
        self.note_tokens = note_tokens

    def __len__(self): return len(self.stay_ids)

    def __getitem__(self, idx):
        sid = self.stay_ids[idx]
        row = self.df.loc[sid]

        # Demographics
        age_t     = torch.tensor([row['age_norm']], dtype=torch.float32)
        marital_t = torch.tensor(max(int(row['marital_status_enc']), 0), dtype=torch.long)
        lang_t    = torch.tensor(max(int(row['language_group_enc']),  0), dtype=torch.long)

        # Time series + per-step labels
        if sid in self.ts_lookup:
            arr = self.ts_lookup[sid]
            T   = arr.shape[0]       # actual steps (already capped at MAX_TS_STEPS)
            if T >= MAX_TS_STEPS:
                # Truncated patient: all steps labeled 1 (never observed discharge)
                ts_t    = torch.from_numpy(arr[:MAX_TS_STEPS])
                ts_mask = torch.zeros(MAX_TS_STEPS, dtype=torch.bool)
                labels  = torch.ones(MAX_TS_STEPS, dtype=torch.long)
            else:
                # Pad to MAX_TS_STEPS
                pad     = np.zeros((MAX_TS_STEPS - T, TS_INPUT_DIM), dtype=np.float32)
                ts_t    = torch.from_numpy(np.concatenate([arr, pad], axis=0))
                ts_mask = torch.zeros(MAX_TS_STEPS, dtype=torch.bool)
                ts_mask[T:] = True
                # steps 0..T-2 -> 1 (in ICU), step T-1 -> 0 (discharge/death), T.. -> -1
                labels  = torch.full((MAX_TS_STEPS,), -1, dtype=torch.long)
                if T > 0:
                    labels[:T - 1] = 1
                    labels[T - 1]  = 0
        else:
            ts_t    = torch.zeros(MAX_TS_STEPS, TS_INPUT_DIM, dtype=torch.float32)
            ts_mask = torch.ones(MAX_TS_STEPS, dtype=torch.bool)
            labels  = torch.full((MAX_TS_STEPS,), -1, dtype=torch.long)

        # Radiology note tokens
        tokens = self.note_tokens.get(sid)
        if tokens is not None:
            input_ids_t = tokens['input_ids'].squeeze(0)
            attn_mask_t = tokens['attention_mask'].squeeze(0)
            has_text_t  = torch.tensor([1.0], dtype=torch.float32)
        else:
            input_ids_t = torch.zeros(TEXT_MAX_LENGTH, dtype=torch.long)
            attn_mask_t = torch.zeros(TEXT_MAX_LENGTH, dtype=torch.long)
            has_text_t  = torch.tensor([0.0], dtype=torch.float32)

        target_t = torch.tensor([row['los_hours']], dtype=torch.float32)
        return {
            'age': age_t, 'marital_enc': marital_t, 'lang_enc': lang_t,
            'ts_seq': ts_t, 'ts_mask': ts_mask, 'step_labels': labels,
            'input_ids': input_ids_t, 'attention_mask': attn_mask_t,
            'has_text': has_text_t, 'target': target_t
        }

print('ICUDataset defined.')


In [8]:
# ── Cell 8: DataLoaders ──────────────────────────────────────────────────
train_dataset = ICUDataset(ids_train, df_discharge, ts_lookup, note_tokens)
val_dataset   = ICUDataset(ids_val,   df_discharge, ts_lookup, note_tokens)
test_dataset  = ICUDataset(ids_test,  df_discharge, ts_lookup, note_tokens)

# num_workers=0 required on Windows
kw = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)
train_loader = DataLoader(train_dataset, shuffle=True,  **kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **kw)
print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val  : {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test : {len(test_dataset)} samples, {len(test_loader)} batches')

Train: 3233 samples, 102 batches
Val  : 693 samples, 22 batches
Test : 693 samples, 22 batches


In [9]:
# ── Cell 9: DemographicEncoder ───────────────────────────────────────────
class DemographicEncoder(nn.Module):
    """
    Encodes age (continuous), marital_status and language_group (categorical).
    Output: (B, DEMO_HIDDEN_DIM)
    """
    def __init__(self, marital_vocab_size, lang_vocab_size,
                 embed_dim=DEMO_EMBED_DIM, hidden_dim=DEMO_HIDDEN_DIM):
        super().__init__()
        # padding_idx=0 gives a zero embedding for unknown categories
        self.marital_embed = nn.Embedding(marital_vocab_size, embed_dim, padding_idx=0)
        self.lang_embed    = nn.Embedding(lang_vocab_size,    embed_dim, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(1 + embed_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU())

    def forward(self, age, marital_enc, lang_enc):
        m = self.marital_embed(marital_enc)   # (B, embed_dim)
        l = self.lang_embed(lang_enc)          # (B, embed_dim)
        return self.fc(torch.cat([age, m, l], dim=1))  # (B, hidden_dim)

print('DemographicEncoder defined.')

DemographicEncoder defined.


In [10]:
# ── Cell 10: PositionalEncoding + TransformerTSEncoder ─────────────────────
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (batch_first)."""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TransformerTSEncoder(nn.Module):
    """
    Causal Transformer encoder for time series.
    Pipeline: Linear(30->embed_dim) -> PosEnc -> CausalTransformerEncoder
    Causal mask: at position t, attention only looks at steps 0..t.
    Padding mask: True positions are ignored by attention.
    Output: (B, MAX_TS_STEPS, TS_EMBED_DIM)  per-step hidden states
    """
    def __init__(self, input_dim=TS_INPUT_DIM, embed_dim=TS_EMBED_DIM,
                 nhead=TS_NHEAD, num_layers=TS_NUM_LAYERS,
                 dim_feedforward=TS_DIM_FF, dropout=TS_DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, embed_dim)
        self.pos_enc    = PositionalEncoding(embed_dim, max_len=512)
        enc = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)

    def forward(self, x, src_key_padding_mask=None):
        # x: (B, T, input_dim)
        x = self.pos_enc(self.input_proj(x))          # (B, T, embed_dim)
        T = x.size(1)
        # Causal mask: upper-triangular -inf so step t only attends to 0..t
        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            T, device=x.device)
        x = self.transformer(x, mask=causal_mask,
                             src_key_padding_mask=src_key_padding_mask)
        return x                                       # (B, T, embed_dim)

print('TransformerTSEncoder defined.')


In [11]:
# ── Cell 11: TextEncoder (Qwen, frozen) ──────────────────────────────────
class TextEncoder(nn.Module):
    """
    Frozen Qwen LM + trainable linear projection.
    Qwen is a causal/decoder-only LM: use mean-pool of last hidden states
    (NOT CLS token -- decoder-only LMs have no CLS token).
    Samples without a note are zeroed out via has_text gate.
    Output: (B, TEXT_PROJ_DIM)
    """
    def __init__(self, model_name=QWEN_MODEL_NAME, proj_dim=TEXT_PROJ_DIM):
        super().__init__()
        print(f'  Loading {model_name} ...')
        self.qwen = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        for p in self.qwen.parameters():
            p.requires_grad = False  # freeze all Qwen weights
        h = self.qwen.config.hidden_size
        self.proj    = nn.Linear(h, proj_dim)  # trainable projection
        self.dropout = nn.Dropout(0.1)
        print(f'  Qwen hidden_size={h} -> proj_dim={proj_dim}')

    def forward(self, input_ids, attention_mask, has_text):
        # torch.no_grad() prevents building autograd graph through frozen Qwen
        with torch.no_grad():
            out = self.qwen(input_ids=input_ids, attention_mask=attention_mask)
        last = out.last_hidden_state  # (B, L, hidden_size)
        mask = attention_mask.unsqueeze(-1).float()
        emb  = (last * mask).sum(1) / mask.sum(1).clamp(min=1)  # mean-pool
        emb  = self.proj(self.dropout(emb))   # (B, proj_dim)
        return emb * has_text  # zero out missing notes

print('TextEncoder defined.')

TextEncoder defined.


In [12]:
# ── Cell 12: FusionModel ────────────────────────────────────────────────────
class FusionModel(nn.Module):
    """
    Per-step ICU-discharge prediction via multi-modal fusion.
    At each timestep t (using only features 0..t via causal masking):
      TS hidden state (B, T, 64) + static demo (B, 64) + static text (B, 128)
      -> concatenate along feature dim -> per-step linear head -> logit
    Output: (B, MAX_TS_STEPS) raw logits
      logit > 0 (prob > 0.5) => patient still in ICU at next step
      logit < 0 (prob < 0.5) => discharge / death predicted
    """
    def __init__(self, marital_vocab_size, lang_vocab_size):
        super().__init__()
        self.demo_enc = DemographicEncoder(marital_vocab_size, lang_vocab_size)
        self.ts_enc   = TransformerTSEncoder()
        self.text_enc = TextEncoder()
        fused = TS_EMBED_DIM + DEMO_HIDDEN_DIM + TEXT_PROJ_DIM  # 64+64+128=256
        self.head = nn.Sequential(
            nn.Linear(fused, FUSION_HIDDEN_DIM), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(FUSION_HIDDEN_DIM, 64), nn.ReLU(),
            nn.Linear(64, 1))                           # per-step logit

    def forward(self, b):
        d  = self.demo_enc(b['age'], b['marital_enc'], b['lang_enc'])  # (B, 64)
        ts = self.ts_enc(b['ts_seq'], b['ts_mask'])                    # (B, T, 64)
        x  = self.text_enc(b['input_ids'], b['attention_mask'],
                           b['has_text'])                              # (B, 128)
        T = ts.size(1)
        # Broadcast static embeddings across all timesteps
        d_exp = d.unsqueeze(1).expand(-1, T, -1)    # (B, T, 64)
        x_exp = x.unsqueeze(1).expand(-1, T, -1)    # (B, T, 128)
        fused  = torch.cat([ts, d_exp, x_exp], dim=-1)   # (B, T, 256)
        logits = self.head(fused).squeeze(-1)              # (B, T)
        return logits

print('FusionModel defined.')


In [13]:
# ── Cell 13: Instantiate Model & Sanity Check ────────────────────────────
model = FusionModel(marital_vocab_size=marital_vocab,
                    lang_vocab_size=lang_vocab).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}')
print(f'Frozen (Qwen)    : {total - trainable:,}')

# Forward-pass sanity check
model.eval()
sb  = next(iter(train_loader))
sbg = {k: v.to(device) for k, v in sb.items()}
with torch.no_grad():
    out = model(sbg)
bs = sb['target'].shape[0]
assert out.shape == (bs, MAX_TS_STEPS), \
    f'Expected ({bs},{MAX_TS_STEPS}), got {tuple(out.shape)}'
print(f'Output shape : {tuple(out.shape)}  (batch x timesteps)')
# Check label coverage
labels     = sb['step_labels']
valid_frac = (labels >= 0).float().mean().item()
print(f'Valid labels fraction per batch: {valid_frac:.3f}')
print('Sanity check passed!')


In [14]:
# ── Cell 14: Training Setup ───────────────────────────────────────────────────
# BCEWithLogitsLoss is handled inside run_epoch (manual masking over valid steps)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=SCHEDULER_PATIENCE, factor=0.5, verbose=True)
print('Criterion : BCEWithLogitsLoss (masked over valid timesteps)')
print(f'Optimizer : Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler : ReduceLROnPlateau (patience={SCHEDULER_PATIENCE}, factor=0.5)')


In [15]:
# ── Cell 15: Training Loop ────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, device, train=True):
    """One epoch. Returns (mean_bce_loss, step_accuracy) over all valid timesteps."""
    model.train(train)
    total_loss, total_correct, total_valid = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            bg     = {k: v.to(device) for k, v in batch.items()}
            logits = model(bg)             # (B, T)
            labels = bg['step_labels']     # (B, T), values 0/1/-1
            valid  = labels >= 0           # (B, T) bool mask
            if valid.sum() == 0:
                continue
            loss = F.binary_cross_entropy_with_logits(
                logits[valid], labels[valid].float())
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    max_norm=1.0)
                optimizer.step()
            n_valid        = valid.sum().item()
            total_loss    += loss.item() * n_valid
            preds_bin      = (logits[valid].detach() > 0).long()
            total_correct += (preds_bin == labels[valid]).sum().item()
            total_valid   += n_valid
    bce = total_loss / max(total_valid, 1)
    acc = total_correct / max(total_valid, 1)
    return bce, acc


train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_loss = float('inf')
print(f'Training for {NUM_EPOCHS} epochs ...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    t_bce, t_acc = run_epoch(model, train_loader, optimizer, device, train=True)
    v_bce, v_acc = run_epoch(model, val_loader,   optimizer, device, train=False)
    scheduler.step(v_bce)
    train_losses.append(t_bce); val_losses.append(v_bce)
    train_accs.append(t_acc);   val_accs.append(v_acc)
    marker = ''
    if v_bce < best_val_loss:
        best_val_loss = v_bce
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        marker = '  <- best'
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train BCE: {t_bce:.4f} Acc: {t_acc:.3f} | '
          f'Val BCE: {v_bce:.4f} Acc: {v_acc:.3f}{marker}')

print(f'\nBest Val BCE: {best_val_loss:.4f}  (saved: {CHECKPOINT_PATH})')


In [16]:
# ── Cell 16: Learning Curve ───────────────────────────────────────────────────
ep = range(1, len(train_losses) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ep, train_losses, label='Train BCE')
ax1.plot(ep, val_losses,   label='Val BCE')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.set_title('BCE Loss over Epochs')
ax1.legend()

ax2.plot(ep, train_accs, label='Train Acc')
ax2.plot(ep, val_accs,   label='Val Acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Per-step Accuracy')
ax2.set_title('Per-step Accuracy over Epochs')
ax2.legend()

plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150)
plt.show()


In [17]:
# ── Cell 17: Test Set Evaluation ──────────────────────────────────────────────────
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

all_true_steps, all_pred_steps = [], []
all_true_los,   all_pred_los   = [], []
all_step_logits, all_step_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        bg      = {k: v.to(device) for k, v in batch.items()}
        logits  = model(bg)                     # (B, T)
        probs   = torch.sigmoid(logits).cpu()   # (B, T)
        labels  = bg['step_labels'].cpu()       # (B, T)
        targets = bg['target'].cpu().squeeze(1) # (B,)

        for i in range(len(targets)):
            valid  = labels[i] >= 0             # (T,)
            T_act  = valid.sum().item()         # actual number of steps
            if T_act == 0:
                continue

            # Collect per-step logits/labels for classification metrics
            all_step_logits.extend(logits[i][valid].cpu().tolist())
            all_step_labels.extend(labels[i][valid].tolist())

            # Predicted discharge: first step where predicted prob < 0.5
            p    = probs[i, :T_act].numpy()
            exit_indices = np.where(p < 0.5)[0]
            if len(exit_indices) > 0:
                pred_step = int(exit_indices[0]) + 1  # steps completed before discharge
            else:
                pred_step = T_act                      # never predicted discharge

            all_true_steps.append(T_act)
            all_pred_steps.append(pred_step)
            all_true_los.append(float(targets[i]))
            # Convert predicted step to hours: step k -> hour_3h = (k-1)*3
            all_pred_los.append(max(0.0, (pred_step - 1) * 3.0))

all_true_steps  = np.array(all_true_steps)
all_pred_steps  = np.array(all_pred_steps)
all_true_los    = np.array(all_true_los)
all_pred_los    = np.array(all_pred_los)
all_step_logits = np.array(all_step_logits)
all_step_labels = np.array(all_step_labels)

# ── Per-step classification metrics ──
step_preds_bin = (all_step_logits > 0).astype(int)
step_acc  = float(np.mean(step_preds_bin == all_step_labels))
step_bce  = float(np.mean(
    np.logaddexp(0, all_step_logits) - all_step_labels * all_step_logits))
pos_rate  = float(all_step_labels.mean())
# True Positive Rate and True Negative Rate
tpr = float(np.mean(step_preds_bin[all_step_labels == 1] == 1))  # sensitivity
tnr = float(np.mean(step_preds_bin[all_step_labels == 0] == 0))  # specificity

# ── Discharge timing estimation metrics ──
step_mae  = float(np.mean(np.abs(all_pred_steps - all_true_steps)))
step_rmse = float(np.sqrt(np.mean((all_pred_steps - all_true_steps) ** 2)))
los_mae   = float(np.mean(np.abs(all_pred_los - all_true_los)))
los_rmse  = float(np.sqrt(np.mean((all_pred_los - all_true_los) ** 2)))
step_corr = float(np.corrcoef(all_pred_steps, all_true_steps)[0, 1])

print('=' * 55)
print('  Per-step Classification Metrics')
print('=' * 55)
print(f'  BCE Loss         : {step_bce:10.4f}')
print(f'  Accuracy         : {step_acc:10.4f}')
print(f'  Sensitivity(TPR) : {tpr:10.4f}  (% in-ICU steps correct)')
print(f'  Specificity(TNR) : {tnr:10.4f}  (% discharge steps correct)')
print(f'  Positive rate    : {pos_rate:10.4f}  (fraction of steps in ICU)')
print('─' * 55)
print('  Discharge Timing Estimation')
print('─' * 55)
print(f'  Step MAE         : {step_mae:10.2f} steps')
print(f'  Step RMSE        : {step_rmse:10.2f} steps')
print(f'  LOS MAE          : {los_mae:10.2f} h')
print(f'  LOS RMSE         : {los_rmse:10.2f} h')
print(f'  Step correlation : {step_corr:10.4f}')
print('=' * 55)

# ── Visualisation ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predicted vs true step count
ax = axes[0]
ax.scatter(all_true_steps, all_pred_steps, alpha=0.3, s=8)
lim = [0, max(all_true_steps.max(), all_pred_steps.max()) + 1]
ax.plot(lim, lim, 'r--', lw=1, label='Perfect')
ax.set_xlabel('True Sequence Length (steps)')
ax.set_ylabel('Predicted Discharge Step')
ax.set_title(f'Pred vs True Length\n(MAE={step_mae:.1f} steps, r={step_corr:.3f})')
ax.legend()

# 2. Abs step error distribution
ax2 = axes[1]
abs_err = np.abs(all_pred_steps - all_true_steps)
ax2.hist(abs_err, bins=40, edgecolor='k', alpha=0.7)
ax2.axvline(float(np.median(abs_err)), color='orange', linestyle='--',
            label=f'Median={np.median(abs_err):.1f}')
ax2.set_xlabel('|Predicted - True| (steps)')
ax2.set_ylabel('Count')
ax2.set_title('Abs Error in Predicted Discharge Step')
ax2.legend()

# 3. Per-step probability distribution by true label
ax3 = axes[2]
probs_flat = 1 / (1 + np.exp(-all_step_logits))
ax3.hist(probs_flat[all_step_labels == 1], bins=40, alpha=0.6,
         label='In ICU (label=1)', color='steelblue')
ax3.hist(probs_flat[all_step_labels == 0], bins=40, alpha=0.6,
         label='Discharge/Death (label=0)', color='salmon')
ax3.axvline(0.5, color='k', linestyle='--', label='threshold=0.5')
ax3.set_xlabel('Predicted P(still in ICU)')
ax3.set_ylabel('Count')
ax3.set_title('Predicted Prob by True Label')
ax3.legend()

plt.tight_layout()
plt.savefig('test_evaluation.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 18: Subgroup Analysis (≤ 7 days vs > 7 days) ───────────────────────────
THRESHOLD_STEPS = int(7 * 24 / 3)  # 56 steps = 7 days at 3-h resolution

mask_lt7 = all_true_steps <= THRESHOLD_STEPS
mask_ge7 = all_true_steps >  THRESHOLD_STEPS

print(f'Subgroup ≤ 7 days (≤{THRESHOLD_STEPS} steps) : n = {mask_lt7.sum()}')
print(f'Subgroup > 7 days  (>{THRESHOLD_STEPS} steps) : n = {mask_ge7.sum()}')
print()

def subgroup_metrics(true_steps, pred_steps, true_los, pred_los, label):
    n       = len(true_steps)
    s_mae   = float(np.mean(np.abs(pred_steps - true_steps)))
    s_rmse  = float(np.sqrt(np.mean((pred_steps - true_steps) ** 2)))
    l_mae   = float(np.mean(np.abs(pred_los - true_los)))
    l_rmse  = float(np.sqrt(np.mean((pred_los - true_los) ** 2)))
    corr    = float(np.corrcoef(pred_steps, true_steps)[0, 1]) if n > 1 else float('nan')
    sep = '=' * 55
    print(sep)
    print(f'  Subgroup: {label}  (n={n})')
    print(sep)
    print(f'  Step MAE         : {s_mae:10.2f} steps')
    print(f'  Step RMSE        : {s_rmse:10.2f} steps')
    print(f'  LOS MAE          : {l_mae:10.2f} h')
    print(f'  LOS RMSE         : {l_rmse:10.2f} h')
    print(f'  Step correlation : {corr:10.4f}')
    return dict(label=label, n=n, s_mae=s_mae, s_rmse=s_rmse,
                l_mae=l_mae, l_rmse=l_rmse, corr=corr)

res_lt7 = subgroup_metrics(
    all_true_steps[mask_lt7], all_pred_steps[mask_lt7],
    all_true_los[mask_lt7],   all_pred_los[mask_lt7],
    label=f'≤ 7 days (≤{THRESHOLD_STEPS} steps)')
print()
res_ge7 = subgroup_metrics(
    all_true_steps[mask_ge7], all_pred_steps[mask_ge7],
    all_true_los[mask_ge7],   all_pred_los[mask_ge7],
    label=f'> 7 days  (>{THRESHOLD_STEPS} steps)')

# ── Visualisation ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Subgroup Evaluation: ≤ 7 days vs > 7 days', fontsize=13)

for row, (mask, res) in enumerate([(mask_lt7, res_lt7), (mask_ge7, res_ge7)]):
    ts = all_true_steps[mask]; ps = all_pred_steps[mask]
    ae = np.abs(ps - ts)

    ax = axes[row, 0]
    ax.scatter(ts, ps, alpha=0.3, s=8)
    lim = [0, max(ts.max(), ps.max()) + 1]
    ax.plot(lim, lim, 'r--', lw=1, label='Perfect')
    ax.set_xlabel('True Steps'); ax.set_ylabel('Predicted Steps')
    ax.set_title(f"{res['label']}\n(MAE={res['s_mae']:.2f} steps, r={res['corr']:.3f})")
    ax.legend()

    ax2 = axes[row, 1]
    ax2.hist(ae, bins=40, edgecolor='k', alpha=0.7)
    ax2.axvline(float(np.median(ae)), color='orange', linestyle='--',
                label=f'Median={np.median(ae):.1f}')
    ax2.set_xlabel('|Pred − True| (steps)'); ax2.set_ylabel('Count')
    ax2.set_title(f"{res['label']} — Abs Error Distribution")
    ax2.legend()

plt.tight_layout()
plt.savefig('subgroup_evaluation.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 19: Kaplan-Meier Survival Curves ────────────────────────────────
# 'Survival' = P(still in ICU at time t)  (event = ICU discharge or death)
# All patients are followed to the end of their ICU stay (no natural censoring).
# KM estimator reduces to 1 - empirical CDF when all events are observed.

df_km = pd.read_csv(DISCHARGE_FILE)   # reload to keep this cell self-contained
# Merge death event info to use unified LOS (death time for ICU deaths)
_df_death_km = pd.read_csv(DEATH_FILE)[['stay_id', 'death_event', 'time_to_icu_death_hours']]
df_km = df_km.merge(_df_death_km, on='stay_id', how='left')
df_km['death_event'] = df_km['death_event'].fillna(0).astype(int)
df_km['los_hours'] = np.where(
    df_km['death_event'] == 1,
    df_km['time_to_icu_death_hours'],
    df_km['time_to_icu_discharge_hours']
)

# ── Kaplan-Meier estimator (no censoring) ────────────────────────────────
def km_curve(durations):
    """
    Returns (times, survival) for a KM curve with no censoring.
    times[0] = 0, survival[0] = 1.0.
    """
    durations = np.sort(durations)
    n = len(durations)
    times     = np.concatenate([[0], durations])
    survival  = np.concatenate([[1.0], 1 - np.arange(1, n + 1) / n])
    return times, survival


def km_ci(durations, alpha=0.05):
    """
    KM survival + 95% CI via Greenwood's formula (log-log transform).
    Returns (times, surv, lower, upper).
    """
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)
    durations = np.sort(durations)
    n = len(durations)
    times    = np.concatenate([[0], durations])
    surv     = np.concatenate([[1.0], 1 - np.arange(1, n + 1) / n])
    # Greenwood variance: Var[S(t)] = S(t)^2 * sum_{i:t_i<=t} d_i/(n_i*(n_i-d_i))
    # With d_i=1 at each distinct time: sum = sum 1/(n_i*(n_i-1)) up to t
    greenwood = np.concatenate([[0.0],
        np.cumsum([1.0 / ((n - i) * (n - i - 1)) if (n - i - 1) > 0 else 0
                   for i in range(n)])])
    # log-log CI
    with np.errstate(divide='ignore', invalid='ignore'):
        log_log_s  = np.log(-np.log(np.where(surv > 0, surv, np.nan)))
        se_loglog  = np.sqrt(greenwood) / np.abs(np.log(np.where(surv > 0, surv, np.nan)))
    lower = np.exp(-np.exp(log_log_s + z * se_loglog))
    upper = np.exp(-np.exp(log_log_s - z * se_loglog))
    lower = np.where(surv <= 0, 0, lower)
    upper = np.where(surv <= 0, 0, upper)
    return times, surv, np.nan_to_num(lower, nan=0), np.nan_to_num(upper, nan=1)


def plot_km(ax, groups, colors, title, xlabel='ICU LOS (hours)',
            max_time=None, show_ci=True, show_median=True):
    """Plot KM curves for multiple groups on ax."""
    if max_time is None:
        max_time = max(g[0].max() for _, g in groups)
    for (label, dur), color in zip(groups, colors):
        t, s, lo, hi = km_ci(dur)
        # Step-function
        ax.step(t, s, where='post', color=color,
                label=f'{label}  (n={len(dur)})', linewidth=1.8)
        if show_ci:
            ax.fill_between(t, lo, hi, step='post', alpha=0.12, color=color)
        # Median
        if show_median:
            idx = np.searchsorted(-s, -0.5)  # first index where s <= 0.5
            if idx < len(t):
                med = t[idx]
                ax.axvline(med, color=color, linestyle=':', linewidth=1, alpha=0.7)
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xlim(0, max_time)
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('P (still in ICU)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.25)


# ── Define stratification groups ──────────────────────────────────────────
t_col = 'los_hours'
MAX_H = 360   # 15 days; truncate x-axis for readability

# Age tertiles
age_cuts = [0, 50, 70, 120]
age_labels = ['Age < 50', 'Age 50-70', 'Age > 70']
df_km['age_group'] = pd.cut(df_km['age'], bins=age_cuts, labels=age_labels)

# Marital status: partnered vs unpartnered
df_km['partnered'] = df_km['marital_status'].map(
    lambda x: 'Married' if x == 'MARRIED' else 'Not married')

# Language
# language_group already in the data

# ── Figure: 4 panels ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Kaplan-Meier Survival Curves — ICU Length of Stay\n'
             '(survival = probability of remaining in ICU)',
             fontsize=13, y=1.01)

PALETTE = ['#2166ac', '#d6604d', '#4dac26', '#8073ac']

# Panel 1: Overall
plot_km(axes[0, 0],
        [('Overall cohort', df_km[t_col].values)],
        [PALETTE[0]],
        title='Overall (n={:,})'.format(len(df_km)),
        max_time=MAX_H)

# Panel 2: By age group
age_groups = [(lbl, df_km.loc[df_km['age_group'] == lbl, t_col].values)
              for lbl in age_labels]
plot_km(axes[0, 1], age_groups, PALETTE[:3],
        title='Stratified by Age Group', max_time=MAX_H)

# Panel 3: By language group
lang_groups = [(lbl, df_km.loc[df_km['language_group'] == lbl, t_col].values)
               for lbl in ['English', 'non-English']]
plot_km(axes[1, 0], lang_groups, PALETTE[:2],
        title='Stratified by Language Group', max_time=MAX_H)

# Panel 4: By marital status
marital_groups = [(lbl, df_km.loc[df_km['partnered'] == lbl, t_col].values)
                  for lbl in ['Married', 'Not married']]
plot_km(axes[1, 1], marital_groups, PALETTE[:2],
        title='Stratified by Marital Status', max_time=MAX_H)

# Annotate medians
for ax in axes.flat:
    ax.text(0.98, 0.55, 'median', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8, color='grey',
            style='italic')

plt.tight_layout()
plt.savefig('km_survival_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: km_survival_curves.png')

# ── Print median LOS per group ────────────────────────────────────────────
print()
print('=== Median ICU LOS (hours) ===')
all_groups = [
    ('Overall',          df_km[t_col]),
    ('Age < 50',         df_km.loc[df_km['age_group'] == 'Age < 50', t_col]),
    ('Age 50-70',        df_km.loc[df_km['age_group'] == 'Age 50-70', t_col]),
    ('Age > 70',         df_km.loc[df_km['age_group'] == 'Age > 70', t_col]),
    ('English',          df_km.loc[df_km['language_group'] == 'English', t_col]),
    ('non-English',      df_km.loc[df_km['language_group'] == 'non-English', t_col]),
    ('Married',          df_km.loc[df_km['partnered'] == 'Married', t_col]),
    ('Not married',      df_km.loc[df_km['partnered'] == 'Not married', t_col]),
]
print(f'  {"Group":<20} {"n":>6}  {"Median (h)":>12}  {"IQR":>18}')
print('  ' + '-' * 60)
for name, series in all_groups:
    n   = len(series)
    med = series.median()
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    print(f'  {name:<20} {n:>6}  {med:>12.1f}  [{q1:.1f}, {q3:.1f}]')


In [ ]:
# ── Cell 20: Model-Predicted vs Actual Survival Curve ────────────────────
# 预测生存函数: S_hat(t) = prod_{k=0}^{t-1} p_k
#   p_k = sigmoid(logit_k) = 模型在第 k 步预测的「还在 ICU」概率
# 将所有测试患者在公共时间网格上的 S_hat(t) 取均值，与真实 KM 曲线对比。
# 依赖: Cell 17（model 已加载）、Cell 19（km_ci 已定义）。

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

PLOT_MAX_H = 360   # x 轴截断（15 天）
STEP_H     = 3     # 每步对应小时数

# ── 按患者收集预测生存曲线 ──────────────────────────────────────────────
patient_surv    = []   # list of (times_h, surv_pred)
test_los_actual = []

with torch.no_grad():
    for batch in test_loader:
        bg      = {k: v.to(device) for k, v in batch.items()}
        logits  = model(bg)                     # (B, T)
        probs   = torch.sigmoid(logits).cpu()   # (B, T)
        labels  = bg['step_labels'].cpu()       # (B, T)
        targets = bg['target'].cpu().squeeze(1) # (B,)

        for i in range(len(targets)):
            valid = labels[i] >= 0
            T_act = valid.sum().item()
            if T_act == 0:
                continue

            p = probs[i, :T_act].numpy()           # (T_act,)
            # S(0)=1, S(t)=prod p[0..t-1]  => shape (T_act+1,)
            surv_pred = np.concatenate([[1.0], np.cumprod(p)])
            # 步骤 k 对应小时: k * STEP_H
            times_h   = np.arange(0, T_act + 1) * STEP_H

            patient_surv.append((times_h, surv_pred))
            test_los_actual.append(float(targets[i]))

# ── 公共时间网格：对所有患者在该网格上插值平均 ───────────────────────────
time_grid       = np.arange(0, PLOT_MAX_H + 1, STEP_H)
surv_accum      = np.zeros(len(time_grid))
surv_count      = np.zeros(len(time_grid))

for times_h, surv_pred in patient_surv:
    for ti, t in enumerate(time_grid):
        # 找到该患者在时间 t 的预测生存值（阶梯插值）
        idx = np.searchsorted(times_h, t, side='right') - 1
        if 0 <= idx < len(surv_pred):
            surv_accum[ti] += surv_pred[idx]
            surv_count[ti] += 1

pred_surv_mean = np.where(surv_count > 0, surv_accum / surv_count, np.nan)

# ── 真实 KM 曲线（测试集）───────────────────────────────────────────────
test_los_arr = np.array(test_los_actual)
t_km, s_km, lo_km, hi_km = km_ci(test_los_arr)
mask_km = t_km <= PLOT_MAX_H

# ── 单图：整体对比 ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.step(t_km[mask_km], s_km[mask_km], where='post',
        color='#2166ac', linewidth=2.2,
        label=f'Actual KM  (n={len(test_los_arr):,})')
ax.fill_between(t_km[mask_km], lo_km[mask_km], hi_km[mask_km],
                step='post', alpha=0.12, color='#2166ac')

ax.plot(time_grid, pred_surv_mean, color='#d6604d', linewidth=2.2,
        linestyle='--', label='Model predicted (mean S̃(t))')

ax.axhline(0.5, color='grey', linestyle=':', linewidth=0.9, alpha=0.7)
ax.set_xlim(0, PLOT_MAX_H)
ax.set_ylim(-0.02, 1.05)
ax.set_xlabel('ICU LOS (hours)', fontsize=12)
ax.set_ylabel('P (still in ICU)', fontsize=12)
ax.set_title('Model-Predicted vs Actual Survival Curve (Test Set)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('model_survival_curve.png', dpi=150)
plt.show()

# ── 按7天分层对比（2个子图）────────────────────────────────────────────────
THRESH_H = 168
mask_short = test_los_arr <  THRESH_H
mask_long  = test_los_arr >= THRESH_H

# 对应患者的预测生存曲线
surv_short, surv_long = [], []
for k, (times_h, sp) in enumerate(patient_surv):
    (surv_short if test_los_arr[k] < THRESH_H else surv_long).append((times_h, sp))

def avg_surv_on_grid(surv_list, grid):
    accum = np.zeros(len(grid)); cnt = np.zeros(len(grid))
    for times_h, sp in surv_list:
        for ti, t in enumerate(grid):
            idx = np.searchsorted(times_h, t, side='right') - 1
            if 0 <= idx < len(sp):
                accum[ti] += sp[idx]; cnt[ti] += 1
    return np.where(cnt > 0, accum / cnt, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model-Predicted vs Actual Survival — Subgroups (Test Set)', fontsize=13)

for ax, mask, surv_list, grp_label, max_h in [
    (axes[0], mask_short, surv_short, f'LOS < 7 days  (n={mask_short.sum()})', THRESH_H),
    (axes[1], mask_long,  surv_long,  f'LOS ≥ 7 days  (n={mask_long.sum()})',  PLOT_MAX_H),
]:
    grid = np.arange(0, max_h + 1, STEP_H)
    pred_m = avg_surv_on_grid(surv_list, grid)

    t_k, s_k, lo_k, hi_k = km_ci(test_los_arr[mask])
    mk = t_k <= max_h
    ax.step(t_k[mk], s_k[mk], where='post', color='#2166ac',
            linewidth=2, label='Actual KM')
    ax.fill_between(t_k[mk], lo_k[mk], hi_k[mk], step='post',
                    alpha=0.12, color='#2166ac')
    ax.plot(grid, pred_m, color='#d6604d', linewidth=2,
            linestyle='--', label='Model predicted')
    ax.axhline(0.5, color='grey', linestyle=':', linewidth=0.8, alpha=0.7)
    ax.set_xlim(0, max_h); ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel('ICU LOS (hours)', fontsize=11)
    ax.set_ylabel('P (still in ICU)', fontsize=11)
    ax.set_title(grp_label, fontsize=12)
    ax.legend(fontsize=10); ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig('model_survival_curve_subgroup.png', dpi=150)
plt.show()
print('Saved: model_survival_curve.png, model_survival_curve_subgroup.png')
